In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 26 — MONITORAMENTO: O GUARDIÃO SILENCIOSO DA PERFORMANCE

> "Lançar um modelo em produção sem monitoramento é como lançar um navio ao mar sem capitão. Ele navega bem por um tempo — mas quando a tempestade vier, ninguém estará no leme."
> — Um Engenheiro de Confiabilidade

O OncoClassify 2.0 estava vivo, encapsulado numa API. Mas o oceano dos dados do mundo real é traiçoeiro. E se a distribuição dos pacientes começasse a mudar? E se um novo scanner produzisse imagens com características ligeiramente diferentes das que usei no treino?

Meu modelo foi treinado num *retrato* do mundo. Mas o mundo é um filme em evolução. Esse fenômeno — o **drift** — é um dos maiores inimigos de um modelo em produção. Se eu não o monitorasse, o desempenho poderia se degradar em silêncio, dos 98% para 95%, para 90%, sem ninguém perceber, até um erro grave. Eu precisava de um guardião.

## Vigiando o Modelo: Detectando Drift

Há dois tipos de *drift*: o **de dados** (a distribuição das *features* de entrada muda) e o **de conceito** (a relação entre *features* e alvo muda). E duas frentes de monitoramento:

**Drift de dados (entrada)** — compara a distribuição dos dados de produção com a de treino. É um **alerta precoce**: detecta a mudança *antes* de o desempenho cair.

**Drift de performance (saída)** — rastreia as métricas ao longo do tempo, mas exige os **rótulos verdadeiros** (que, na medicina, podem demorar dias ou semanas).

Vamos simular o drift de dados com três métricas complementares, aplicadas a uma *feature* na qual injetamos uma mudança artificial (inflar `worst radius` em 15%, como se um equipamento novo superestimasse a medida).

#### Código 26.1: Detecção de Drift (KS, Wasserstein e PSI)


In [ ]:
import numpy as np, matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import ks_2samp, wasserstein_distance
from oncoclassify_utils import X_train

# "producao" simulada = uma COPIA perturbada da distribuicao de treino.
# (Nao usamos o X_test aqui -- o cofre so sera aberto no Capitulo 24.)
producao = X_train.copy()
producao["worst radius"] = producao["worst radius"] * 1.15
ref, prod = X_train["worst radius"], producao["worst radius"]

# 1) Kolmogorov-Smirnov: teste de hipotese (p<0.05 -> distribuicoes diferentes)
ks_stat, p_valor = ks_2samp(ref, prod)
# 2) Wasserstein: "custo" de transformar uma distribuicao na outra (magnitude)
w = wasserstein_distance(ref, prod)
# 3) PSI: indice de estabilidade populacional, com faixas de referencia consagradas
def psi(esperado, atual, n_faixas=10):
    cortes = np.percentile(esperado, np.linspace(0, 100, n_faixas + 1))
    cortes[0] -= 1e-6; cortes[-1] += 1e-6
    e = np.clip(np.histogram(esperado, cortes)[0] / len(esperado), 1e-4, None)
    a = np.clip(np.histogram(atual,     cortes)[0] / len(atual),     1e-4, None)
    return float(np.sum((e - a) * np.log(e / a)))

print(f"KS: estatistica={ks_stat:.4f}  p-valor={p_valor:.2e}")
print(f"Wasserstein: {w:.4f}")
print(f"PSI: {psi(ref.values, prod.values):.4f}")

# Quantas das 30 features acusam drift? (so alteramos uma)
n_drift = sum(ks_2samp(X_train[c], producao[c]).pvalue < 0.05 for c in X_train.columns)
print(f"Features com drift detectado (p<0.05): {n_drift} de 30")

sns.kdeplot(ref, fill=True, alpha=0.5, label="Referência (treino)")
sns.kdeplot(prod, fill=True, alpha=0.5, label="Produção (com drift)")
plt.title("Data Drift em 'worst radius'"); plt.legend(); plt.tight_layout(); plt.show()

KS: estatistica=0.2583  p-valor=1.76e-14
Wasserstein: 2.4252
PSI: 0.3892
Features com drift detectado (p<0.05): 1 de 30


![Drift em worst radius](media/figura_26_1.png)

As três métricas soam o alarme, cada uma à sua maneira:

**KS** dá um veredicto binário com significância estatística: p-valor de 1,8×10⁻¹⁴ — rejeitamos com folga a hipótese de que as distribuições são iguais.

**Wasserstein** (2,43) quantifica a *magnitude* do deslocamento — útil para priorizar qual *drift* investigar primeiro.

**PSI** (0,39) tem faixas consagradas: < 0,1 estável; 0,1–0,25 mudança moderada; **> 0,25 mudança severa**. 0,39 confirma um *drift* grave, exigindo ação (retreinamento).

E um detalhe importante: **apenas 1 das 30 *features*** acusou *drift* — exatamente a que alteramos. O monitor é **específico**, não dá alarme onde não há mudança.

> **📘 PARA SABER — Qual métrica usar?**
>
> Use o **KS** como alerta rápido e automatizado (barato, binário). Se ele disparar, calcule **Wasserstein** e **PSI** para medir a severidade e priorizar. O PSI é o preferido em setores regulados (crédito, seguros) por suas faixas de decisão claras. Uma abordagem híbrida — KS diário em todas as *features*, e Wasserstein/PSI sob demanda — cobre bem os dois mundos.

> **📌 CHECKLIST DO CAPÍTULO 26**
>
> ✅ Entende **data drift** e **concept drift** e por que ameaçam modelos em produção.
>
> ✅ Distingue monitoramento de **entrada** (alerta precoce) e de **performance** (exige rótulos).
>
> ✅ Executou o Código 26.1 e detectou o *drift* com KS, Wasserstein e PSI.
>
> ✅ Notou que o monitor é **específico** (só a *feature* alterada disparou).
>
> **⚠️ CRÍTICO:** Monitoramento não é um "extra": é parte essencial da operação responsável. Sem ele, você está voando às cegas.

O monitoramento completou minha visão do ciclo de vida do OncoClassify 2.0: não um projeto com fim, mas um sistema contínuo de *deployment*, observação e adaptação. Eu tinha um plano não só para construir o modelo, mas para mantê-lo saudável ao longo do tempo.

Com a arquitetura técnica e operacional no lugar, minha atenção se voltou para uma dimensão diferente, mas igualmente crítica: a **ética**. Meu modelo, treinado num dataset de uma certa população, funcionaria igualmente bem para todos os grupos? Um modelo 98% preciso em média, mas 85% para um subgrupo, não é justo. Eu precisava investigar a **equidade**.
